# Fine Tuning FLAN T5 on DialogSum with LoRA



In [1]:
# Personal parameters

SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)


SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3


## Environment Setup

In [2]:
%pip install -q --upgrade "torchao>=0.16.0" transformers datasets peft accelerate sentencepiece evaluate rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


After running the installation cell for the first time, restart the kernel if your environment previously had an older version of torchao loaded. Then run the notebook from the beginning.


In [3]:
import os
import random
import json
import time
import platform
import numpy as np
import pandas as pd
import torch
import torchao
import transformers
import peft

from packaging.version import Version
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, TaskType, get_peft_model

assert Version(torchao.__version__) >= Version("0.16.0"), (
    f"torchao {torchao.__version__} is too old. "
    "Restart the kernel after running the installation cell."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("torchao:", torchao.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU not detected. Training will be slower.")


Python: 3.13.15
PyTorch: 2.11.0+cu128
torchao: 0.18.0
Transformers: 5.17.0
PEFT: 0.21.1
Device: cuda
GPU: Tesla T4


## Task 1: Data Preparation

DialogSum contains conversations paired with human written summaries. The dialogue is used as the model input and the summary is used as the target.


In [4]:
dataset = load_dataset("knkarthick/dialogsum")

print(dataset)
print()
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))
print("Columns:", dataset["train"].column_names)


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Train size: 12460
Validation size: 500
Test size: 1500
Columns: ['id', 'dialogue', 'summary', 'topic']


In [5]:
# Use SLICE as the reproducible starting position.

TRAIN_SIZE = 1000
VAL_SIZE = 100

train_start = SLICE
train_end = min(train_start + TRAIN_SIZE, len(dataset["train"]))

val_start = SLICE
val_end = min(val_start + VAL_SIZE, len(dataset["validation"]))

train_dataset = dataset["train"].select(range(train_start, train_end))
val_dataset = dataset["validation"].select(range(val_start, val_end))

print("SLICE:", SLICE)
print("Training subset size:", len(train_dataset))
print("Validation subset size:", len(val_dataset))


SLICE: 46
Training subset size: 1000
Validation subset size: 100


In [6]:
def format_sample(example):
    return {
        "input": "Summarize the following dialogue:\n\n" + example["dialogue"],
        "target": example["summary"],
    }

processed_samples = train_dataset.map(format_sample)

for i in range(2):
    print("=" * 80)
    print("PROCESSED SAMPLE", i + 1)
    print()
    print("INPUT")
    print(processed_samples[i]["input"])
    print()
    print("TARGET")
    print(processed_samples[i]["target"])
    print()


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

PROCESSED SAMPLE 1

INPUT
Summarize the following dialogue:

#Person1#: Mister Ewing said we should show up at the conference center at 4 o'clock, right?
#Person2#: Yes, he specially asked us not to be late. Some of the people from our East York branch office are coming and he wants to make a good impression on them. How are you getting there?
#Person1#: I was thinking of taking my car but I think I'm just going to take the underground because there is construction on the highway. What about you?
#Person2#: I'll be taking the underground as well. Why don't we go together? I've been to the conference center only once, and I'm not sure if I can find my way around there.

TARGET
To get to the conference centre early, #Person1# and #Person2# decide to take the underground, and #Person2# proposes to go there together.

PROCESSED SAMPLE 2

INPUT
Summarize the following dialogue:

#Person1#: What's the accommodation like in London Ahmed?
#Person2#: Well, it's a bit difficult. There are thousa

## Tokenization

In [7]:
MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Tokenizer loaded: google/flan-t5-small


In [8]:
def preprocess_function(examples):
    prompts = [
        "Summarize the following dialogue:\n\n" + dialogue
        for dialogue in examples["dialogue"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print(tokenized_train)
print(tokenized_val)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


## Task 2: Baseline Inference

Two fixed examples from the test split are selected now. These exact dialogues will also be used after fine tuning so the comparison is controlled.


In [9]:
test_dataset = dataset["test"]
eval_indices = [0, 1]

evaluation_examples = [
    {
        "dialogue": test_dataset[i]["dialogue"],
        "reference": test_dataset[i]["summary"],
    }
    for i in eval_indices
]

for i, example in enumerate(evaluation_examples, 1):
    print("=" * 80)
    print("EVALUATION EXAMPLE", i)
    print()
    print("DIALOGUE")
    print(example["dialogue"])
    print()
    print("REFERENCE SUMMARY")
    print(example["reference"])
    print()


EVALUATION EXAMPLE 1

DIALOGUE
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, pl

In [10]:
def generate_summary(model, dialogue):
    prompt = "Summarize the following dialogue:\n\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )


baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

baseline_outputs = [
    generate_summary(baseline_model, example["dialogue"])
    for example in evaluation_examples
]

for i in range(2):
    print("=" * 80)
    print("EXAMPLE", i + 1)
    print()
    print("BASELINE OUTPUT")
    print(baseline_outputs[i])
    print()
    print("REFERENCE")
    print(evaluation_examples[i]["reference"])
    print()


model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

EXAMPLE 1

BASELINE OUTPUT
                                                               

REFERENCE
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

EXAMPLE 2

BASELINE OUTPUT
                                                               

REFERENCE
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.



In [11]:
os.makedirs("outputs", exist_ok=True)

baseline_records = [
    {
        "example": i + 1,
        "dialogue": evaluation_examples[i]["dialogue"],
        "reference": evaluation_examples[i]["reference"],
        "baseline": baseline_outputs[i],
    }
    for i in range(2)
]

with open("outputs/baseline_outputs.json", "w", encoding="utf8") as f:
    json.dump(baseline_records, f, indent=2, ensure_ascii=False)

print("Baseline outputs saved.")


Baseline outputs saved.


In [12]:
del baseline_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Task 3: Apply LoRA

The initial experiment uses rank 4. Alpha and dropout are stated explicitly as required by the assignment.


In [13]:
LORA_RANK = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("LoRA rank:", LORA_RANK)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("Target modules: q, v")
print()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print()

model.print_trainable_parameters()


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA rank: 4
LoRA alpha: 16
LoRA dropout: 0.05
Target modules: q, v

Total parameters: 77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%

trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


## Task 4: Fine Tuning

This first training run uses LoRA rank 4. The training logs and final loss are retained as evidence of the run.


In [14]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./training_output_r4",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Starting LoRA rank 4 training")
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time
final_training_loss = train_result.training_loss

print()
print(f"Final training loss: {final_training_loss:.6f}")
print(f"Training time: {training_time:.2f} seconds")


Starting LoRA rank 4 training
Training examples: 1000
Validation examples: 100


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan



Final training loss: 0.000000
Training time: 85.12 seconds


In [15]:
adapter_dir = "./saved_adapters/dialogsum_r4"

os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Saved LoRA adapter to:", adapter_dir)


Saved LoRA adapter to: ./saved_adapters/dialogsum_r4


## Task 5: Post Fine Tuning Inference

The same two dialogues from the baseline are used again. Generation remains intentionally simple at this stage so later decoding experiments have a meaningful starting point.


In [16]:
r4_outputs = [
    generate_summary(model, example["dialogue"])
    for example in evaluation_examples
]

for i in range(2):
    print("=" * 80)
    print("EXAMPLE", i + 1)
    print()
    print("BEFORE FINE TUNING")
    print(baseline_outputs[i])
    print()
    print("AFTER FINE TUNING, LoRA r=4")
    print(r4_outputs[i])
    print()
    print("REFERENCE")
    print(evaluation_examples[i]["reference"])
    print()


EXAMPLE 1

BEFORE FINE TUNING
                                                               

AFTER FINE TUNING, LoRA r=4
                                                               

REFERENCE
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

EXAMPLE 2

BEFORE FINE TUNING
                                                               

AFTER FINE TUNING, LoRA r=4
                                                               

REFERENCE
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.



## Initial Output Comparison

In [17]:
comparison_df = pd.DataFrame(
    {
        "Example": [1, 2],
        "Reference": [x["reference"] for x in evaluation_examples],
        "Baseline": baseline_outputs,
        "LoRA r=4": r4_outputs,
    }
)

pd.set_option("display.max_colwidth", None)
comparison_df


,Example,Reference,Baseline,LoRA r=4
0,1,Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.,,
1,2,"In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.",,


In [18]:
all_outputs = [
    {
        "example": i + 1,
        "dialogue": evaluation_examples[i]["dialogue"],
        "reference": evaluation_examples[i]["reference"],
        "baseline": baseline_outputs[i],
        "lora_r4": r4_outputs[i],
    }
    for i in range(2)
]

with open("outputs/initial_r4_results.json", "w", encoding="utf8") as f:
    json.dump(all_outputs, f, indent=2, ensure_ascii=False)

comparison_df.to_csv(
    "outputs/initial_r4_comparison.csv",
    index=False,
)

run_summary = {
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "model": MODEL_NAME,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "training_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "final_training_loss": float(final_training_loss),
    "training_time_seconds": float(training_time),
}

with open("outputs/initial_run_metrics.json", "w", encoding="utf8") as f:
    json.dump(run_summary, f, indent=2)

print(json.dumps(run_summary, indent=2))


{
  "SID4": 2046,
  "SEED": 2046,
  "SLICE": 46,
  "model": "google/flan-t5-small",
  "lora_rank": 4,
  "lora_alpha": 16,
  "lora_dropout": 0.05,
  "total_parameters": 77133184,
  "trainable_parameters": 172032,
  "training_examples": 1000,
  "validation_examples": 100,
  "final_training_loss": 0.0,
  "training_time_seconds": 85.12366771697998
}
